### Create Input Data
1. Extract valid samples;
2. Design the efficient features.

In [1]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
else:
    print("GPU not detected. Please check PyTorch installation.")

PyTorch version: 2.6.0+cu124
CUDA available: True
GPU Name: NVIDIA GeForce RTX 4080


In [2]:
%pip install pympi-ling pandas numpy

Note: you may need to restart the kernel to use updated packages.


In [13]:
import pympi
import pandas as pd
import numpy as np

#### Explore the Raw Data

In [22]:
# --- 1. SET FILE PATHS ---
# Specify paths for one group to verify the logic
eaf_path = '/home/ist-seminar/Documents/data/annotation/meta-segment/20251027_g2_all_meta_combined.eaf'
csv_s1_path = '/home/ist-seminar/Documents/data/20251027/pose_20251027_g1_s1_meta_enc.csv'
csv_s2_path = '/home/ist-seminar/Documents/data/20251027/pose_20251027_g1_s2_meta_enc.csv'

# --- 2. LOAD FILES ---
eaf = pympi.Eaf(eaf_path)
df_s1 = pd.read_csv(csv_s1_path)
df_s2 = pd.read_csv(csv_s2_path)

# --- 3. GET ANNOTATIONS ---
# Extract start time, end time, and label from 'meta-topic' tier
# Times are in milliseconds (ms)
tier_data = eaf.get_annotation_data_for_tier('meta-topic')

print(f"Total annotations found: {len(tier_data)}\n")

# --- 4. DATA SLICING AND COGNITION CHECK ---
for start_ms, end_ms, label in tier_data:
    # Convert ms to seconds to match CSV 'time' column
    start_sec = start_ms / 1000
    end_sec = end_ms / 1000
    
    # Extract the category name (e.g., "Main-Th")
    category = label.split('/')[-1]
    
    # Slice the CSV dataframes
    slice_s1 = df_s1[(df_s1['time'] >= start_sec) & (df_s1['time'] <= end_sec)]
    slice_s2 = df_s2[(df_s2['time'] >= start_sec) & (df_s2['time'] <= end_sec)]
    
    # Print summary for visualization
    print(f"Label: {category:<10} | Duration: {end_sec - start_sec:>5.2f}s | Rows: {len(slice_s1)}")
    
    # peek at the first 5 rows of the sliced data
    print(slice_s1[['time','pitch', 'yaw', 'roll']].head(5))
    print(slice_s2[['time','pitch', 'yaw', 'roll']].head(5))
    print("-" * 30)

Total annotations found: 74

Label:  Other     | Duration:  1.72s | Rows: 52
       time  pitch     yaw   roll
0  0.000000  2.844 -11.703 -2.944
1  0.033367  2.844 -11.703 -2.944
2  0.066733  2.844 -11.703 -2.944
3  0.100100  2.844 -11.703 -2.944
4  0.133467  2.844 -11.703 -2.944
       time  pitch     yaw   roll
0  0.000000  7.854  13.748 -0.346
1  0.033367  7.854  13.748 -0.346
2  0.066733  7.854  13.748 -0.346
3  0.100100  7.854  13.748 -0.346
4  0.133467  7.854  13.748 -0.346
------------------------------
Label:  Main-Th   | Duration:  5.44s | Rows: 163
           time  pitch     yaw   roll
1263  42.142100  2.682 -15.030  1.424
1264  42.175467  2.563 -15.092  1.441
1265  42.208833  2.226 -14.802  1.493
1266  42.242200  2.002 -14.890  1.519
1267  42.275567  2.208 -14.761  1.607
           time   pitch    yaw   roll
1263  42.142100  18.104  5.846 -3.312
1264  42.175467  18.949  5.553 -3.406
1265  42.208833  19.994  6.111 -3.087
1266  42.242200  20.585  6.414 -2.831
1267  42.275567  

#### Feature Engineering

In [28]:
# --- 1. CONFIGURATION ---
# Define groups and their respective files
data_groups = [
    {'id': 'g1', 
     'eaf': '/home/ist-seminar/Documents/data/annotation/meta-segment/20251027_g1_all_meta_combined.eaf', 
     's1': '/home/ist-seminar/Documents/data/20251027/pose_20251027_g1_s1_meta_enc.csv', 
     's2': '/home/ist-seminar/Documents/data/20251027/pose_20251027_g1_s2_meta_enc.csv'},
    {'id': 'g2', 
     'eaf': '/home/ist-seminar/Documents/data/annotation/meta-segment/20251027_g2_all_meta_combined.eaf', 
     's1': '/home/ist-seminar/Documents/data/20251027/pose_20251027_g2_s1_meta_enc.csv', 
     's2': '/home/ist-seminar/Documents/data/20251027/pose_20251027_g2_s2_meta_enc.csv'},
    {'id': 'g3', 
     'eaf': '/home/ist-seminar/Documents/data/annotation/meta-segment/20251027_g3_all_meta_combined.eaf', 
     's1': '/home/ist-seminar/Documents/data/20251027/pose_20251027_g3_s1_meta_enc.csv', 
     's2': '/home/ist-seminar/Documents/data/20251027/pose_20251027_g3_s2_meta_enc.csv'}
]

# --- 2. CORE FEATURE CALCULATOR ---
def get_features(df_s1, df_s2, label, group_id):
    """Calculate comprehensive stats and interaction features for a segment."""
    feats = {'group': group_id, 'label': label}
    
    # Pre-calculate common length for interaction metrics
    min_len = min(len(df_s1), len(df_s2))
    
    for col in ['pitch', 'yaw', 'roll']:
        # 1. Individual Activity (How much they move)
        feats[f's1_{col}_std'] = df_s1[col].std()
        feats[f's2_{col}_std'] = df_s2[col].std()
        
        # 2. Pose Bias (Where they are looking/leaning on average)
        # Low abs mean in Yaw suggests looking at the monitor
        feats[f's1_{col}_mean'] = df_s1[col].mean()
        feats[f's2_{col}_mean'] = df_s2[col].mean()

        # 3. Interaction Metrics
        s1_vals = df_s1[col].values[:min_len]
        s2_vals = df_s2[col].values[:min_len]
        
        # Inter-personal distance (e.g., Yaw diff indicates mutual gaze)
        feats[f'inter_{col}_diff_mean'] = np.mean(np.abs(s1_vals - s2_vals))
        
        # Synchrony (Pearson Correlation)
        # Checks if S1 and S2 move their heads together (e.g., nodding in style evaluation)
        if min_len > 10:
            corr = np.corrcoef(s1_vals, s2_vals)[0, 1]
            feats[f'inter_{col}_corr'] = corr if not np.isnan(corr) else 0
        else:
            feats[f'inter_{col}_corr'] = 0
            
    return feats

# --- 3. PROCESSING PIPELINE ---
def process_collection(raw_data):
    """Iterate through groups, slice data, and aggregate features."""
    all_rows = []
    for rd in raw_data:
        eaf = pympi.Eaf(rd['eaf'])
        csv1, csv2 = pd.read_csv(rd['s1']), pd.read_csv(rd['s2'])
        
        for start_ms, end_ms, raw_label in eaf.get_annotation_data_for_tier('meta-topic'):
            # Clean label (e.g., "topic/Main-Th" -> "Main-Th")
            label = raw_label.split('/')[-1].strip()
            
            # Slice CSVs
            s, e = start_ms / 1000, end_ms / 1000
            seg1 = csv1[(csv1['time'] >= s) & (csv1['time'] <= e)]
            seg2 = csv2[(csv2['time'] >= s) & (csv2['time'] <= e)]
            
            if len(seg1) > 5 and len(seg2) > 5:
                all_rows.append(get_features(seg1, seg2, label, rd['id']))
    return pd.DataFrame(all_rows)

# --- 4. RUN AND SAVE ---
df_all = process_collection(data_groups)
    
print(f"All samples: {df_all.shape[0]} samples")
df_all.to_csv('dataset.csv', index=False)

All samples: 195 samples


### Train ML Models

In [18]:
%pip install scikit-learn seaborn matplotlib

  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 50.6 MB/s  0:00:00
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 46.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 42.7 MB/s  0:00:00
Using cached pyparsing-3.3.2-py3-none-any.whl (122 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [seaborn]m6/7 [seaborn]ib]
Note: you may need to restart the kernel to use updated packages.


In [29]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import LeaveOneGroupOut
import numpy as np

# 1. Prepare features (X), labels (y), and groups
X = df_all.drop(columns=['label', 'group']).fillna(0)
y = df_all['label']
groups = df_all['group']

# 2. Initialize Leave-One-Group-Out and model
logo = LeaveOneGroupOut()
rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)

all_preds = []
all_true = []

# 3. Cross-validation loop
for train_idx, test_idx in logo.split(X, y, groups):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    # Train and predict
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)
    
    all_preds.extend(y_pred)
    all_true.extend(y_test)
    
    # Print accuracy per fold
    acc = accuracy_score(y_test, y_pred)
    print(f"Group {groups.iloc[test_idx[0]]} accuracy: {acc:.3f}")

# 4. Final evaluation report
print("\n### FINAL AGGREGATED REPORT ###")
print(classification_report(all_true, all_preds, zero_division=0))

# 5. Global feature importance
rf.fit(X, y)
importances = sorted(zip(rf.feature_importances_, X.columns), reverse=True)
print("\n### TOP 5 FEATURES ###")
for score, name in importances[:5]:
    print(f"{name}: {score:.4f}")

Group g1 accuracy: 0.136
Group g2 accuracy: 0.446
Group g3 accuracy: 0.226

### FINAL AGGREGATED REPORT ###
              precision    recall  f1-score   support

    Main-Con       0.17      0.03      0.05        34
  Main-Style       0.00      0.00      0.00         5
     Main-Th       0.36      0.60      0.45        90
    Meta-Con       0.00      0.00      0.00        56
       Other       0.00      0.00      0.00        10

    accuracy                           0.28       195
   macro avg       0.11      0.13      0.10       195
weighted avg       0.20      0.28      0.22       195


### TOP 5 FEATURES ###
s2_yaw_mean: 0.0749
s1_pitch_std: 0.0748
inter_yaw_diff_mean: 0.0717
s1_roll_mean: 0.0664
inter_pitch_diff_mean: 0.0641


In [30]:
print("Train Label Distribution:\n", y_train.value_counts())
print("Test Label Distribution:\n", y_test.value_counts())

Train Label Distribution:
 label
Meta-Con      56
Main-Th       41
Main-Con      28
Main-Style     5
Other          3
Name: count, dtype: int64
Test Label Distribution:
 label
Main-Th     49
Other        7
Main-Con     6
Name: count, dtype: int64


### Analysis

1. The model attempts to guess **Main-Th**? Since this type of data are much more than others.  
-> Any other models that can cope with imbalanced and insufficent data ?
2. We may need **more powerful** features since the importances of all features are extremely low.  
-> Design more useful features from POSE DATA by carefully analyzing the video *(Current stage)*  
-> Its very difficult to 
-> Introduce other modalities ?
